In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [6]:

class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        """
        alpha: balancing factor for class imbalance (can be a tensor for per-class weighting)
        gamma: focusing parameter (higher = more focus on hard examples)
        reduction: 'mean' (default), 'sum', or 'none'
        """
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        """
        inputs: predicted logits (not probabilities), shape (batch_size, num_classes)
        targets: ground-truth class labels, shape (batch_size,)
        """
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')  # Compute Cross Entropy Loss
        print("ce_loss", ce_loss)
        pt = torch.exp(-ce_loss)  # Compute p_t (probability of true class)
        print("pt: ", pt)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss  # Apply focal loss formula

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss  # 'none' keeps the original shape

In [7]:
# Example Usage
criterion = FocalLoss(alpha=0.25, gamma=2)  # Adjust alpha & gamma as needed
logits = torch.randn(5, 3, requires_grad=True)  # Simulated predictions (batch_size=5, num_classes=3)
labels = torch.tensor([1, 0, 2, 1, 0])  # True labels

loss = criterion(logits, labels)
print(loss)

ce_loss tensor([1.9077, 2.2758, 0.9769, 3.1663, 1.6395], grad_fn=<NllLossBackward0>)
pt:  tensor([0.1484, 0.1027, 0.3765, 0.0422, 0.1941], grad_fn=<ExpBackward0>)
tensor(0.3783, grad_fn=<MeanBackward0>)
